# Module 03 — Exercises: Autograd
Exercises 1–3 extend a small `Value` engine. Exercises 4–6 use PyTorch autograd.

In [ ]:
# Standard setup: make the repo root importable, pick a device, fix the random seed.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]  # notebooks live in modules/XX_name/, so go up two levels
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common.device import get_device, seed_everything, fast_dev_run

FAST_DEV_RUN = fast_dev_run()  # True only for automated checks; leave it False when learning
DATA_DIR = ROOT / "data"
device = get_device()
seed_everything(42)

In [ ]:
import math
import torch


class Value:
    """Minimal engine from the lesson (add, mul, pow only). You'll add more operations."""
    def __init__(self, data, _prev=()):
        self.data, self.grad, self._prev, self._backward = float(data), 0.0, set(_prev), lambda: None

    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data + o.data, (self, o))
        def _b():
            self.grad += out.grad; o.grad += out.grad
        out._backward = _b
        return out

    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data * o.data, (self, o))
        def _b():
            self.grad += o.data * out.grad; o.grad += self.data * out.grad
        out._backward = _b
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,))
        def _b():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _b
        return out

    __radd__ = __add__
    __rmul__ = __mul__
    def __neg__(self): return self * -1
    def __sub__(self, o): return self + (-o)
    def __truediv__(self, o): return self * (o if isinstance(o, Value) else Value(o)) ** -1

    def backward(self):
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for p in v._prev:
                    build(p)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

### Exercise 1 — Add `exp` and `log`
Implement `value_exp(v)` and `value_log(v)` returning new `Value`s with correct `_backward` closures.
d/dx eˣ = eˣ, d/dx ln x = 1/x.

In [ ]:
def value_exp(v: Value) -> Value:
    # TODO
    raise NotImplementedError


def value_log(v: Value) -> Value:
    # TODO
    raise NotImplementedError

In [ ]:
x = Value(0.7)
y = value_log(value_exp(x) * x)        # = x + ln x  -> dy/dx = 1 + 1/x
y.backward()
assert math.isclose(y.data, 0.7 + math.log(0.7))
assert math.isclose(x.grad, 1 + 1 / 0.7)
print("Exercise 1 passed ✅")

### Exercise 2 — Sigmoid via composition (no new backward needed)
Write `value_sigmoid(v)` using only operations that already have backward rules: `1 / (1 + exp(-v))`.
Autograd then gives the gradient for free.

In [ ]:
def value_sigmoid(v: Value) -> Value:
    # TODO
    raise NotImplementedError

In [ ]:
x = Value(0.4)
s = value_sigmoid(x); s.backward()
sig = 1 / (1 + math.exp(-0.4))
assert math.isclose(s.data, sig) and math.isclose(x.grad, sig * (1 - sig))
print("Exercise 2 passed ✅")

### Exercise 3 — Binary cross-entropy with your engine
Logistic regression on two features: `p = sigmoid(w1*x1 + w2*x2 + b)`.
Compute the BCE for **one** example, `-(y*log(p) + (1-y)*log(1-p))`, then call `backward()`.
Check that `w1.grad == (p - y) * x1` (the formula from module 02).

In [ ]:
w1, w2, b = Value(0.5), Value(-1.0), Value(0.1)
x1, x2, y_true = 2.0, 1.0, 1.0
p = None     # TODO
loss = None  # TODO
loss.backward()

In [ ]:
assert math.isclose(w1.grad, (p.data - y_true) * x1, rel_tol=1e-9)
assert math.isclose(b.grad, (p.data - y_true), rel_tol=1e-9)
print("Exercise 3 passed ✅")

### Exercise 4 — Gradients with PyTorch
For $f(x) = \sum_i \sin(x_i)\,x_i^2$, use autograd to return ∂f/∂x for a given 1-D tensor (it should **not** need
`requires_grad` set by the caller). Compare with the analytic answer `cos(x)·x² + 2x·sin(x)`.

In [ ]:
def grad_f(x: torch.Tensor) -> torch.Tensor:
    # TODO: clone x, enable grad, compute f, backward, return the gradient
    raise NotImplementedError

In [ ]:
x = torch.linspace(-2, 2, 7)
assert torch.allclose(grad_f(x), torch.cos(x) * x ** 2 + 2 * x * torch.sin(x), atol=1e-5)
print("Exercise 4 passed ✅")

### Exercise 5 — Fit a quadratic with autograd
Fit `y ≈ a·x² + b·x + c` by gradient descent with raw tensors (no `nn` and no `optim`). Remember to update inside
`torch.no_grad()` and to reset the grads. Reach an MSE below 0.02.

In [ ]:
torch.manual_seed(0)
xs = torch.linspace(-2, 2, 100)
ys = 1.5 * xs ** 2 - 0.8 * xs + 0.3 + 0.1 * torch.randn(100)
params = torch.zeros(3, requires_grad=True)     # [a, b, c]
# TODO: training loop

In [ ]:
with torch.no_grad():
    mse = ((params[0] * xs ** 2 + params[1] * xs + params[2] - ys) ** 2).mean().item()
print("params:", params.detach().numpy().round(3), "mse:", round(mse, 4))
assert mse < 0.02
assert torch.allclose(params.detach(), torch.tensor([1.5, -0.8, 0.3]), atol=0.1)
print("Exercise 5 passed ✅")

### Exercise 6 — A custom autograd Function: straight-through rounding
`round()` has zero gradient almost everywhere, which would block learning. The *straight-through estimator* uses
rounding in the forward pass but pretends it was the identity in the backward pass. (Quantized networks and the
VQ-VAE use this trick.) Implement `forward` (round) and `backward` (pass the gradient through unchanged).

In [ ]:
class RoundSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # TODO
        raise NotImplementedError

    @staticmethod
    def backward(ctx, grad_out):
        # TODO
        raise NotImplementedError

In [ ]:
x = torch.tensor([0.2, 1.7, -2.6], requires_grad=True)
y = RoundSTE.apply(x)
(y * torch.tensor([1.0, 2.0, 3.0])).sum().backward()
assert torch.equal(y.detach(), torch.tensor([0.0, 2.0, -3.0]))
assert torch.equal(x.grad, torch.tensor([1.0, 2.0, 3.0]))
print("Exercise 6 passed ✅")